In [2]:
import math
import pathlib
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F

# Find the repo root, so `from jabarti_llm...` works wherever this is opened.
ROOT = pathlib.Path.cwd()
while not (ROOT / "jabarti_llm" / "config.py").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

torch.manual_seed(0)

print("torch     :", torch.__version__)
print("repo root :", ROOT)

torch     : 2.13.0+cpu
repo root : r:\Developing\IT\LLM-Projects\jabarti-llm-from-scratch


In [4]:
from jabarti_llm.config import ModelConfig

TOKENS = ["the", "cat", "sat", "on", "mat"]

cfg = ModelConfig(
    vocab_size=32,
    d_model=8,
    n_heads=2,
    n_layers=2,
    max_seq_len=16,
    dropout=0.1,
)

B = 1
T = len(TOKENS)

x = torch.randn(B, T, cfg.d_model)

print("d_model    :", cfg.d_model)
print("n_heads    :", cfg.n_heads, " -> d_k =", cfg.d_k)
print("n_layers   :", cfg.n_layers)
print("d_ff       :", cfg.d_ff, " <- __post_init__ filled this in: 4 * d_model")
print("vocab_size :", cfg.vocab_size)
print("max_seq_len:", cfg.max_seq_len)

d_model    : 8
n_heads    : 2  -> d_k = 4
n_layers   : 2
d_ff       : 3072  <- __post_init__ filled this in: 4 * d_model
vocab_size : 32
max_seq_len: 16


---

# Part 1 -- What we already have, and what is missing

Two modules are finished and we will not rebuild them:

| module | built in | what it does |
|---|---|---|
| `MultiHeadAttention` | `attention.py`, previous notebook | mixes information **across tokens** |
| `FeedForward` | `feedforward.py` | computes **per token**, after the mixing |

Both have the same signature in and out: `(B, T, d_model) -> (B, T, d_model)`.

So the obvious thing to do is just call one after the other.

In [6]:
from jabarti_llm.model.attention import MultiHeadAttention
from jabarti_llm.model.feedforward import FeedForward

attn = MultiHeadAttention(cfg)
ffn = FeedForward(cfg)

attn_out, updated_cache = attn(x)
y = ffn(attn_out)

print("x     :", tuple(x.shape))
print("attn  :", tuple(attn_out.shape))
print("ffn   :", tuple(y.shape))
print()

print("same shape in and out?", tuple(y.shape) == tuple(x.shape))

x     : (1, 5, 8)
attn  : (1, 5, 8)
ffn   : (1, 5, 8)

same shape in and out? True


---
# The residual: give the gradient a way home

```
Residual = "keep the original, add the change on top."
```



Training works by pushing a gradient backwards through every layer. Each layer
**multiplies** the gradient by something on the way past.

Multiply enough numbers smaller than 1 together and you get zero. That is not a
metaphor -- `float32` really does reach exactly `0.0`. Let us watch it happen.

The experiment: stack `depth` feed-forward sublayers, ask for the gradient at
the **input**, and print how much of it survived the trip.

---

Example
```
gradient norm arriving back at the input

  depth     plain stack   with residual
      1       8.151e-03       2.827e+00
      2       2.503e-05       2.824e+00
      5       3.859e-13       2.818e+00
     10       0.000e+00       2.834e+00
     20       0.000e+00       2.822e+00
     30       0.000e+00       2.824e+00
```

## How ?

It is one line of calculus. For a sublayer `f`:

```
plain:      y = f(x)          ->   dy/dx = f'(x)
residual:   y = x + f(x)      ->   dy/dx = 1 + f'(x)
                                        ^
                                        this
```

Stacking multiplies these together. In the plain stack you multiply `f'` by `f'`
by `f'`... and if each one is around 0.5, thirty of them is `0.5^30`, which is
about one in a billion.

In the residual stack every factor is **`1 + something`**. The `1` is an
unbroken path from the loss straight back to the input -- the *highway*. The
sublayer's contribution is a *detour* off that highway. Even if every `f'` is
zero, the gradient still arrives intact through the `1`s.

---

The residual fixed the backward pass. It made the **forward** pass worse.

Look at what the stack now computes:

```
x1 = x0 + f1(x0)
x2 = x1 + f2(x1)
x3 = x2 + f3(x2)
...

```

---

Nothing ever gets smaller. Every layer *adds*. If a sublayer's output tends to be
a bit larger than its input, layer 2 receives something bigger than layer 1 did,
so it outputs something bigger still, and the growth compounds.

**LayerNorm** is the reset. For each token vector separately it subtracts the
mean and divides by the standard deviation, so whatever comes in, what comes out
has mean 0 and std 1.


In [8]:
ln = nn.LayerNorm(cfg.d_model)

wild_input = torch.tensor([
    [
        [50.0, 52.0, 48.0, 51.0, 49.0, 53.0, 47.0, 50.0]
    ]
])

normed_input = ln(wild_input)

print(wild_input.shape)
print(normed_input.shape)


torch.Size([1, 1, 8])
torch.Size([1, 1, 8])


In [11]:
print("wild_input: ")
print("  mean:", round(wild_input.mean().item(), 3),
      "  std:", round(wild_input.std(unbiased=False).item(), 3))
print()

print("normed_input: ")
print("  mean:", round(normed_input.mean().item(), 3),
      "  std:", round(normed_input.std(unbiased=False).item(), 3))
print()

wild_input: 
  mean: 50.0   std: 1.871

normed_input: 
  mean: 0.0   std: 1.0



---
# -- Pre-norm vs post-norm: *where* the norm goes

We now have a residual and a LayerNorm. There are two ways to combine them, and
the choice is not cosmetic.

```
post-norm  (the 2017 original)     x = LayerNorm( x + Attention(x) )
pre-norm   (GPT-2, and this repo)  x = x + Attention( LayerNorm(x) )
```
